# Value-head fine-tune — Phase 2

Full-scale training on **all** Stockfish-eval positions (~13M) for 3 epochs, starting from the Phase 1 checkpoint (`value_finetune_phase1_0.076.pt`, MSE 0.076, Pearson r 0.83).

**Target metrics:** Pearson r 0.90–0.93, prediction std 0.47–0.49, MSE 0.04–0.05, policy KL near zero.

A full diagnostic (MSE, Pearson r, prediction mean/std, policy KL) is printed and a checkpoint saved every 4M positions processed (~10 checkpoints total).

## 1. Imports and configuration

In [ ]:
import copy
import math
import sys
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.amp.autocast_mode import autocast
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

# Resolve project root regardless of whether the kernel starts in training/ or repo root
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "training" / "train.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from training.train import ChessTransformer

# --- Paths ---
CHECKPOINT_PATH = PROJECT_ROOT / "models" / "value_finetune_phase1_0.076.pt"
DATA_PATH       = PROJECT_ROOT / "data" / "processed" / "stockfish_eval_dataset_full.pt"
MODELS_DIR      = PROJECT_ROOT / "models"

# --- Hyperparameters (identical to Phase 1 except dataset scale and EPOCHS) ---
VAL_FRACTION  = 0.05            # ~5% val split
BATCH_SIZE    = 640
ACCUM_STEPS   = 3               # effective batch = 640 x 3 = 1920
EPOCHS        = 3
VALUE_LR      = 1e-4            # value_head params
BACKBONE_LR   = 1e-5            # everything else
WEIGHT_DECAY  = 1e-4
WARMUP_STEPS  = 200             # warmup-then-constant schedule
GRAD_CLIP     = 1.0

USE_KL_REG = True
KL_WEIGHT  = 1.0

CHECKPOINT_INTERVAL = 4_000_000   # full diagnostic + save every N positions processed
DRIFT_BATCH_SIZE    = 5000

SEED               = 42
DATALOADER_WORKERS = 6

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load Phase 1 checkpoint

In [ ]:
print(f"Loading Phase 1 checkpoint: {CHECKPOINT_PATH}")
assert CHECKPOINT_PATH.exists(), f"checkpoint not found: {CHECKPOINT_PATH}"
ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
print(f"Checkpoint keys: {list(ckpt.keys())}")
print(f"Phase 1 final_value_loss:  {ckpt.get('final_value_loss', float('nan')):.4f}")
print(f"Phase 1 final_kl:          {ckpt.get('final_kl', float('nan')):.6e}")

model = ChessTransformer()
model.load_state_dict(ckpt["model_state_dict"])
model = model.to(DEVICE)

# Architecture sanity check
assert model.embedding.num_embeddings == 43, "vocab_size != 43"
assert model.embedding.embedding_dim == 512, "d_model != 512"
assert len(model.transformer.layers) == 8,   "num_layers != 8"
assert model.head_dim == 64,                  "head_dim != 64"

num_params = sum(p.numel() for p in model.parameters())
print(f"Model params: {num_params:,}")
print("Loaded Phase 1 weights \u2014 Phase 2 will train with a fresh optimizer.")

## 3. Frozen reference copy for KL self-distillation

In [ ]:
ref_model = copy.deepcopy(model).eval()
for p in ref_model.parameters():
    p.requires_grad_(False)
ref_model = ref_model.to(DEVICE)

ref_grad_count = sum(1 for p in ref_model.parameters() if p.requires_grad)
print(f"Reference model on {DEVICE}: {sum(p.numel() for p in ref_model.parameters()):,} params, "
      f"{ref_grad_count} with requires_grad (should be 0)")

if DEVICE.type == "cuda":
    print(f"CUDA memory allocated: {torch.cuda.memory_allocated() / 1024**2:,.0f} MB")

## 4. Load full Stockfish-labeled dataset

Expects `stockfish_eval_dataset_full.pt` to already exist. If it doesn't, run:

```
python data/eval_data_processing.py --full
```

from the project root to build it from `data/chessData.csv`.

In [ ]:
assert DATA_PATH.exists(), (
    f"dataset not found: {DATA_PATH}\n"
    f"Build it with:  python data/eval_data_processing.py --full"
)
print(f"Loading full dataset: {DATA_PATH}")
print(f"  size on disk: {DATA_PATH.stat().st_size / 1024**3:.2f} GB")
data = torch.load(DATA_PATH, weights_only=True)
assert set(data.keys()) >= {"tokens", "values"}, f"missing expected keys: {data.keys()}"

tokens  = data["tokens"]
values  = data["values"]
n_total = tokens.size(0)

print(f"\nDataset: {n_total:,} positions")
print(f"  tokens: {tokens.shape}  dtype={tokens.dtype}")
print(f"  values: {values.shape}  dtype={values.dtype}")

assert values.min() >= -1.0 and values.max() <= 1.0, "values outside [-1, 1]"

v_np = values.numpy()
p05, p50, p95 = np.percentile(v_np, [5, 50, 95])
print("\nValue distribution (tanh-transformed Stockfish eval, white perspective):")
print(f"  range:       [{v_np.min():.4f}, {v_np.max():.4f}]")
print(f"  mean / std:  {v_np.mean():+.4f} / {v_np.std():.4f}")
print(f"  p05/p50/p95: {p05:+.4f} / {p50:+.4f} / {p95:+.4f}")

## 5. Train/val split

In [ ]:
g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(n_total, generator=g)

n_val   = int(n_total * VAL_FRACTION)
n_train = n_total - n_val

train_tokens = tokens[perm[:n_train]]
train_values = values[perm[:n_train]]
val_tokens   = tokens[perm[n_train:]]
val_values   = values[perm[n_train:]]

print(f"Train: {n_train:,} | Val: {n_val:,}")
print(f"Effective batch size: {BATCH_SIZE * ACCUM_STEPS:,}")
total_positions_to_process = n_train * EPOCHS
print(f"Total positions across {EPOCHS} epochs: {total_positions_to_process / 1e6:.1f}M")
estimated_ckpts = math.ceil(total_positions_to_process / CHECKPOINT_INTERVAL)
print(f"Expected checkpoints: ~{estimated_ckpts} (one every {CHECKPOINT_INTERVAL / 1e6:.0f}M positions)")

## 6. Evaluation helpers

In [ ]:
@torch.no_grad()
def measure_value_loss(m, tokens_t, values_t, batch_size=BATCH_SIZE):
    m.eval()
    total_sq_err = 0.0
    total = 0
    for i in range(0, tokens_t.size(0), batch_size):
        x = tokens_t[i:i+batch_size].to(DEVICE, dtype=torch.long,    non_blocking=True)
        y = values_t[i:i+batch_size].to(DEVICE, dtype=torch.float32, non_blocking=True)
        with autocast(device_type=DEVICE.type, dtype=torch.bfloat16):
            _, pred = m(x)
        total_sq_err += F.mse_loss(pred.float(), y, reduction="sum").item()
        total += x.size(0)
    return total_sq_err / total


@torch.no_grad()
def measure_policy_kl(m, ref, tokens_t, batch_size=BATCH_SIZE):
    # tokens_t must already be on DEVICE; returns mean per-position KL(current || reference)
    m.eval()
    kl_sum = 0.0
    total  = 0
    for i in range(0, tokens_t.size(0), batch_size):
        x = tokens_t[i:i+batch_size]
        with autocast(device_type=DEVICE.type, dtype=torch.bfloat16):
            cur_logits, _ = m(x)
            ref_logits, _ = ref(x)
        cur_log_probs = F.log_softmax(cur_logits.float(), dim=-1)
        ref_probs     = F.softmax(ref_logits.float(),     dim=-1)
        kl_sum += F.kl_div(cur_log_probs, ref_probs, reduction="sum").item()
        total  += x.size(0)
    return kl_sum / total


@torch.no_grad()
def full_diagnostic(m, ref, val_tok, val_val, drift_tok, label=""):
    # MSE, Pearson r, pred mean/std on full val split + policy KL on drift batch.
    # Sets m.eval() internally; restores m.train() before returning.
    m.eval()

    preds_buf   = []
    targets_buf = []
    for i in range(0, val_tok.size(0), BATCH_SIZE):
        x = val_tok[i:i+BATCH_SIZE].to(DEVICE, dtype=torch.long, non_blocking=True)
        y = val_val[i:i+BATCH_SIZE]
        with autocast(device_type=DEVICE.type, dtype=torch.bfloat16):
            _, pred = m(x)
        preds_buf.append(pred.float().cpu())
        targets_buf.append(y)

    preds   = torch.cat(preds_buf)
    targets = torch.cat(targets_buf)

    mse       = F.mse_loss(preds, targets).item()
    pearson_r = torch.corrcoef(torch.stack([preds, targets]))[0, 1].item()
    pred_mean = preds.mean().item()
    pred_std  = preds.std().item()
    kl_val    = measure_policy_kl(m, ref, drift_tok)

    header = f"  {label}  " if label else ""
    print("=" * 60)
    print(header)
    print(f"  MSE:              {mse:.4f}   (target: 0.04\u20130.05)")
    print(f"  Pearson r:        {pearson_r:.4f}   (target: 0.90\u20130.93)")
    print(f"  Prediction mean:  {pred_mean:+.4f}")
    print(f"  Prediction std:   {pred_std:.4f}   (target: 0.47\u20130.49)")
    print(f"  Policy KL:        {kl_val:.6e}   (should stay near 0)")
    print("=" * 60)

    m.train()
    return dict(mse=mse, pearson_r=pearson_r,
                pred_mean=pred_mean, pred_std=pred_std, policy_kl=kl_val)


print("Evaluation helpers defined.")

## 7. Baseline diagnostic

MSE and policy KL at Phase 2 start (= Phase 1 final state). Policy KL vs the frozen reference is zero at baseline since the models are identical.

In [ ]:
# Fixed drift batch on GPU — same positions used at every checkpoint
drift_n      = min(DRIFT_BATCH_SIZE, n_val)
drift_tokens = val_tokens[:drift_n].to(DEVICE, dtype=torch.long, non_blocking=True)

print("Measuring Phase 2 baseline (= Phase 1 final state)...")
baseline_metrics = full_diagnostic(
    model, ref_model, val_tokens, val_values, drift_tokens,
    label="BASELINE (Phase 1 final)",
)
baseline_mse = baseline_metrics["mse"]
print(f"\nbaseline MSE: {baseline_mse:.4f}  (Phase 1 ended at 0.076)")

## 8. DataLoaders, parameter groups, optimizer, scheduler

In [ ]:
train_ds = TensorDataset(train_tokens, train_values)
val_ds   = TensorDataset(val_tokens,   val_values)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    pin_memory=True, num_workers=DATALOADER_WORKERS,
    persistent_workers=True, prefetch_factor=4, drop_last=False,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    pin_memory=True, num_workers=DATALOADER_WORKERS,
    persistent_workers=True, prefetch_factor=4, drop_last=False,
)

value_head_params = list(model.value_head.parameters())
value_head_ids    = {id(p) for p in value_head_params}
backbone_params   = [p for p in model.parameters() if id(p) not in value_head_ids]
print(f"Value head params: {sum(p.numel() for p in value_head_params):,}")
print(f"Backbone params:   {sum(p.numel() for p in backbone_params):,}")

optimizer = optim.AdamW(
    [
        {"params": value_head_params, "lr": VALUE_LR},
        {"params": backbone_params,   "lr": BACKBONE_LR},
    ],
    weight_decay=WEIGHT_DECAY,
)

num_batches               = len(train_loader)
optimizer_steps_per_epoch = math.ceil(num_batches / ACCUM_STEPS)
total_optimizer_steps     = optimizer_steps_per_epoch * EPOCHS


def warmup_const_then_decay(step):
    # Warmup -> full LR through epochs 1-2 -> cosine decay to 10% across epoch 3.
    # Keeps the Phase 1 pattern of training at full LR, then cools down only at
    # the end so the optimizer can settle out of its noise floor.
    if step < WARMUP_STEPS:
        return 0.1 + 0.9 * (step / max(1, WARMUP_STEPS))
    decay_start = 2 * optimizer_steps_per_epoch  # start of final epoch
    if step < decay_start:
        return 1.0
    progress = (step - decay_start) / max(1, total_optimizer_steps - decay_start)
    return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress))


scheduler = LambdaLR(optimizer, warmup_const_then_decay)

print(f"Train batches: {num_batches:,} | Optimizer steps/epoch: {optimizer_steps_per_epoch:,}")
print(f"Total optimizer steps: {total_optimizer_steps:,}")
print(f"Effective batch: {BATCH_SIZE * ACCUM_STEPS:,} | Warmup: {WARMUP_STEPS} steps")
print(f"LR schedule: warmup {WARMUP_STEPS} steps -> constant through step "
      f"{2 * optimizer_steps_per_epoch:,} -> cosine decay to 10% by step {total_optimizer_steps:,}")

## 9. Training loop

bfloat16 autocast for forward passes (same as `train.py`). Losses recomputed in fp32 for stability. KL gradient flows only through the current model — `ref_model` is queried under `torch.no_grad()`.

Every `CHECKPOINT_INTERVAL` positions processed: print a full diagnostic and save a checkpoint.

In [ ]:
log_steps       = []
log_value_loss  = []
log_kl_loss     = []
log_total_loss  = []

ckpt_log: list[dict] = []
positions_processed  = 0
next_checkpoint_at   = CHECKPOINT_INTERVAL
ckpt_count           = 0

step = 0
optimizer.zero_grad(set_to_none=True)

model.train()
ref_model.eval()

training_start  = time.time()
accum_value_sum = accum_kl_sum = accum_total_sum = 0.0
accum_count = 0

for epoch in range(EPOCHS):
    pbar = tqdm(
        enumerate(train_loader),
        total=num_batches,
        desc=f"Epoch {epoch + 1}/{EPOCHS}",
    )
    for batch_idx, (tok, tgt_v) in pbar:
        tok   = tok.to(DEVICE,   dtype=torch.long,    non_blocking=True)
        tgt_v = tgt_v.to(DEVICE, dtype=torch.float32, non_blocking=True)

        with autocast(device_type=DEVICE.type, dtype=torch.bfloat16):
            cur_logits, pred_v = model(tok)
            if USE_KL_REG:
                with torch.no_grad():
                    ref_logits, _ = ref_model(tok)

        value_loss = F.mse_loss(pred_v.float(), tgt_v)

        if USE_KL_REG:
            cur_log_probs = F.log_softmax(cur_logits.float(), dim=-1)
            ref_probs     = F.softmax(ref_logits.float(),     dim=-1)
            kl_loss    = F.kl_div(cur_log_probs, ref_probs, reduction="batchmean")
            total_loss = value_loss + KL_WEIGHT * kl_loss
        else:
            kl_loss    = torch.zeros((), device=DEVICE)
            total_loss = value_loss

        (total_loss / ACCUM_STEPS).backward()

        positions_processed += tok.size(0)
        accum_value_sum += value_loss.item()
        accum_kl_sum    += kl_loss.item()
        accum_total_sum += total_loss.item()
        accum_count     += 1

        is_accum_boundary = (batch_idx + 1) % ACCUM_STEPS == 0
        is_last_batch     = (batch_idx + 1) == num_batches

        if is_accum_boundary or is_last_batch:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            step += 1

            if step % 200 == 0 or step == 1:
                avg_v = accum_value_sum / accum_count
                avg_k = accum_kl_sum    / accum_count
                avg_t = accum_total_sum / accum_count
                lr_v  = optimizer.param_groups[0]["lr"]
                lr_b  = optimizer.param_groups[1]["lr"]
                log_steps.append(step)
                log_value_loss.append(avg_v)
                log_kl_loss.append(avg_k)
                log_total_loss.append(avg_t)
                pbar.set_postfix(
                    step=step, V=f"{avg_v:.4f}", KL=f"{avg_k:.4f}",
                    lr_v=f"{lr_v:.1e}", lr_b=f"{lr_b:.1e}",
                    pos_M=f"{positions_processed/1e6:.1f}",
                )

            accum_value_sum = accum_kl_sum = accum_total_sum = 0.0
            accum_count = 0

            # ── periodic checkpoint ──
            while positions_processed >= next_checkpoint_at:
                ckpt_count += 1
                pos_M = positions_processed / 1e6
                elapsed = time.time() - training_start

                print(f"\n{'='*60}")
                print(f"Checkpoint {ckpt_count:2d}  |  {pos_M:.1f}M positions  "
                      f"|  epoch {epoch + 1}/{EPOCHS}  |  step {step:,}  "
                      f"|  elapsed {elapsed/3600:.2f}h")
                metrics = full_diagnostic(
                    model, ref_model, val_tokens, val_values, drift_tokens,
                    label=f"Checkpoint {ckpt_count:2d} \u2014 {pos_M:.1f}M pos processed",
                )
                print()

                save_name = (
                    f"value_finetune_phase2_ckpt{ckpt_count:02d}_{int(pos_M)}M.pt"
                )
                save_path = MODELS_DIR / save_name
                torch.save(
                    {
                        "checkpoint_num":      ckpt_count,
                        "positions_processed": positions_processed,
                        "epoch":               epoch + 1,
                        "step":                step,
                        "model_state_dict":    model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "scheduler_state_dict": scheduler.state_dict(),
                        "metrics":             metrics,
                        "phase1_checkpoint":   str(CHECKPOINT_PATH),
                        "config": {
                            "VAL_FRACTION":         VAL_FRACTION,
                            "BATCH_SIZE":           BATCH_SIZE,
                            "ACCUM_STEPS":          ACCUM_STEPS,
                            "EPOCHS":               EPOCHS,
                            "VALUE_LR":             VALUE_LR,
                            "BACKBONE_LR":          BACKBONE_LR,
                            "WEIGHT_DECAY":         WEIGHT_DECAY,
                            "USE_KL_REG":           USE_KL_REG,
                            "KL_WEIGHT":            KL_WEIGHT,
                            "WARMUP_STEPS":         WARMUP_STEPS,
                            "CHECKPOINT_INTERVAL":  CHECKPOINT_INTERVAL,
                            "SEED":                 SEED,
                            "n_train":              n_train,
                            "n_val":                n_val,
                        },
                    },
                    save_path,
                )
                print(f"Saved checkpoint -> {save_path.name}")

                ckpt_log.append({"ckpt": ckpt_count, "pos_M": pos_M,
                                 "epoch": epoch + 1, "step": step,
                                 **metrics})

                model.train()  # full_diagnostic sets eval; restore train mode
                next_checkpoint_at += CHECKPOINT_INTERVAL

train_time = time.time() - training_start
print(f"\nTraining complete in {train_time/3600:.2f}h ({step:,} optimizer steps)")

## 10. Final validation

In [ ]:
print("Measuring final metrics on full val split...")
final_metrics = full_diagnostic(
    model, ref_model, val_tokens, val_values, drift_tokens,
    label="FINAL \u2014 end of Phase 2",
)

print("\nPhase 2 summary:")
print(f"  Baseline MSE:     {baseline_metrics['mse']:.4f}  (Phase 1 final)")
print(f"  Final MSE:        {final_metrics['mse']:.4f}  (target: 0.04\u20130.05)")
print(f"  Baseline r:       {baseline_metrics['pearson_r']:.4f}")
print(f"  Final r:          {final_metrics['pearson_r']:.4f}  (target: 0.90\u20130.93)")
print(f"  Final pred std:   {final_metrics['pred_std']:.4f}  (target: 0.47\u20130.49)")
print(f"  Final policy KL:  {final_metrics['policy_kl']:.6e}  (should be near 0)")

final_save_path = MODELS_DIR / f"value_finetune_phase2_final_{final_metrics['mse']:.3f}.pt"
torch.save(
    {
        "epoch":               EPOCHS,
        "positions_processed": positions_processed,
        "step":                step,
        "model_state_dict":    model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "baseline_metrics":    baseline_metrics,
        "final_metrics":       final_metrics,
        "phase1_checkpoint":   str(CHECKPOINT_PATH),
        "config": {
            "VAL_FRACTION":         VAL_FRACTION,
            "BATCH_SIZE":           BATCH_SIZE,
            "ACCUM_STEPS":          ACCUM_STEPS,
            "EPOCHS":               EPOCHS,
            "VALUE_LR":             VALUE_LR,
            "BACKBONE_LR":          BACKBONE_LR,
            "WEIGHT_DECAY":         WEIGHT_DECAY,
            "USE_KL_REG":           USE_KL_REG,
            "KL_WEIGHT":            KL_WEIGHT,
            "WARMUP_STEPS":         WARMUP_STEPS,
            "CHECKPOINT_INTERVAL":  CHECKPOINT_INTERVAL,
            "SEED":                 SEED,
            "n_train":              n_train,
            "n_val":                n_val,
        },
    },
    final_save_path,
)
print(f"\nFinal model saved -> {final_save_path}")

## 11. Checkpoint progression table

In [ ]:
print(f"{'Ckpt':>5}  {'Pos (M)':>8}  {'Epoch':>5}  {'Step':>7}  {'MSE':>7}  {'r':>7}  "
      f"{'Pred std':>9}  {'Pred mean':>10}  {'Policy KL':>11}")
print("-" * 85)
for row in ckpt_log:
    print(f"{row['ckpt']:>5}  {row['pos_M']:>8.1f}  {row['epoch']:>5}  {row['step']:>7,}  "
          f"{row['mse']:>7.4f}  {row['pearson_r']:>7.4f}  "
          f"{row['pred_std']:>9.4f}  {row['pred_mean']:>+10.4f}  "
          f"{row['policy_kl']:>11.4e}")
print("-" * 85)
print(f"{'Final':>5}  {positions_processed/1e6:>8.1f}  {EPOCHS:>5}  {step:>7,}  "
      f"{final_metrics['mse']:>7.4f}  {final_metrics['pearson_r']:>7.4f}  "
      f"{final_metrics['pred_std']:>9.4f}  {final_metrics['pred_mean']:>+10.4f}  "
      f"{final_metrics['policy_kl']:>11.4e}")

## 12. Training trajectory plots

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(log_steps, log_value_loss, label="train value loss (MSE)", linewidth=0.8)
axes[0].axhline(baseline_metrics["mse"], color="gray",  linestyle="--",
                label=f"baseline ({baseline_metrics['mse']:.3f})")
axes[0].axhline(final_metrics["mse"],    color="green", linestyle=":",
                label=f"final val ({final_metrics['mse']:.3f})")
axes[0].axhline(0.045, color="orange", linestyle=":", alpha=0.6, label="target mid (0.045)")
axes[0].set_xlabel("optimizer step")
axes[0].set_ylabel("MSE")
axes[0].set_title("Value loss vs Stockfish targets")
# Mark checkpoint positions
for i, row in enumerate(ckpt_log):
    axes[0].axvline(row["step"], color="blue", alpha=0.2, linewidth=0.7,
                    label="checkpoint" if i == 0 else None)
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].plot(log_steps, log_kl_loss, color="tab:orange", linewidth=0.8,
             label="train KL (per batch)")
axes[1].axhline(final_metrics["policy_kl"], color="green", linestyle=":",
                label=f"final val KL ({final_metrics['policy_kl']:.4f})")
for i, row in enumerate(ckpt_log):
    axes[1].axvline(row["step"], color="blue", alpha=0.2, linewidth=0.7,
                    label="checkpoint" if i == 0 else None)
axes[1].set_xlabel("optimizer step")
axes[1].set_ylabel("KL divergence")
axes[1].set_title("Policy drift (KL vs frozen Phase 1 reference)")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()